# DINOv2: Predict on Entire Images

Runs dead-tree segmentation on full aerial images (rather than pre-tiled HuggingFace dataset tiles). For each input image the pipeline:
1. Tiles the image into 256×256 patches with `patchify`
2. Runs DINOv2 inference on every patch
3. Stitches predicted masks back into a full-resolution mask
4. Stitches the cropped RGB patches back into a matching RGB image

Outputs a predicted mask PNG and a cropped RGB PNG for each input image.

Requires a saved, trained DINOv2 model -> see `DINOv2Model.ipynb`.

## Configuration

Update the paths below before running.

In [ ]:
import glob

# --- Input: directory containing GeoTIFF images to process ---
INPUT_DIR  = '/explore/nobackup/people/sking11/FullTransect'
IMAGE_GLOB = '*.tif'  # pattern to match image files

# --- Output directories (created automatically if they don't exist) ---
OUTPUT_MASK_DIR  = '/explore/nobackup/people/sking11/FullTransectPredictedMasks'
OUTPUT_IMAGE_DIR = '/explore/nobackup/people/sking11/FullTransectPatchifiedImages'

# --- Model ---
MODEL_PATH = '/explore/nobackup/people/sking11/dinov2model_6400.pth'

# --- Tiling ---
PATCH_SIZE = 256   # tile size in pixels
BATCH_SIZE = 4     # images per forward pass

# Collect all input images
image_paths = sorted(glob.glob(f'{INPUT_DIR}/{IMAGE_GLOB}'))
print(f'Found {len(image_paths)} image(s) to process:')
for p in image_paths:
    print(' ', p)

## 1. Imports

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from patchify import patchify
from tqdm import tqdm
import albumentations as A
from transformers import Dinov2Model, Dinov2PreTrainedModel
from transformers.modeling_outputs import SemanticSegmenterOutput

## 2. Model Definition

Must match the architecture used during training in `DINOv2Model.ipynb`.

In [ ]:
class LinearClassifier(nn.Module):
    def __init__(self, in_channels, tokenW=32, tokenH=32, num_labels=1):
        super().__init__()
        self.in_channels = in_channels
        self.width       = tokenW
        self.height      = tokenH
        self.classifier  = nn.Conv2d(in_channels, num_labels, (1, 1))

    def forward(self, embeddings):
        embeddings = embeddings.reshape(-1, self.height, self.width, self.in_channels)
        embeddings = embeddings.permute(0, 3, 1, 2)
        return self.classifier(embeddings)


class Dinov2ForSemanticSegmentation(Dinov2PreTrainedModel):
    def __init__(self, config, class_weights=None):
        super().__init__(config)
        self.dinov2      = Dinov2Model(config)
        self.classifier  = LinearClassifier(config.hidden_size, 32, 32, config.num_labels)
        self.class_weights = class_weights

    def forward(self, pixel_values, output_hidden_states=False, output_attentions=False, labels=None):
        outputs          = self.dinov2(pixel_values,
                                       output_hidden_states=output_hidden_states,
                                       output_attentions=output_attentions)
        patch_embeddings = outputs.last_hidden_state[:, 1:, :]  # exclude CLS token
        batch_size, num_patches, embedding_dim = patch_embeddings.shape
        expected = self.classifier.width * self.classifier.height
        if num_patches != expected:
            raise ValueError(f'Unexpected patch count: {num_patches}, expected {expected}')
        logits = self.classifier(patch_embeddings)
        logits = nn.functional.interpolate(logits, size=pixel_values.shape[2:],
                                           mode='bilinear', align_corners=False)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss(weight=self.class_weights)(logits, labels)
        return SemanticSegmenterOutput(
            loss=loss, logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions
        )

## 3. Tiling and Stitching Utilities

`tile_image` splits a full image into a flat list of patches and returns the grid dimensions so they can be reassembled later.

`unpatchify_mask` and `unpatchify_rgb` reassemble patches back into a full image. Because `patchify` may crop a few pixels from the right/bottom edge to achieve an exact grid, the stitched output may be slightly smaller than the original.

In [ ]:
ADE_MEAN = np.array([123.675, 116.280, 103.530]) / 255
ADE_STD  = np.array([58.395,  57.120,  57.375])  / 255

patch_transform = A.Compose([
    A.Resize(width=448, height=448),
    A.Normalize(mean=ADE_MEAN.tolist(), std=ADE_STD.tolist()),
])


def tile_image(image_path, patch_size=256):
    """Load an image, tile it into (patch_size x patch_size) patches.
    Returns:
        patches_rgb : np.ndarray of shape (N, patch_size, patch_size, 3) — raw RGB patches
        num_rows    : int — number of patch rows
        num_cols    : int — number of patch columns
    """
    image = np.array(Image.open(image_path).convert('RGB'))
    grid  = patchify(image, (patch_size, patch_size, 3), step=patch_size)
    num_rows, num_cols = grid.shape[0], grid.shape[1]
    patches_rgb = grid.reshape(-1, patch_size, patch_size, 3)  # (N, H, W, 3)
    return patches_rgb, num_rows, num_cols


class PatchDataset(Dataset):
    """Wraps a flat array of RGB patches and applies the normalisation transform."""
    def __init__(self, patches_rgb, transform):
        self.patches   = patches_rgb
        self.transform = transform

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        patch = self.patches[idx]  # (H, W, 3) uint8
        transformed = self.transform(image=patch)['image']  # still (H, W, 3)
        tensor = torch.tensor(transformed).float().permute(2, 0, 1)  # (3, H, W)
        return tensor


def unpatchify(patches_list, patch_size, num_rows, num_cols, channels=None):
    """Stitch a flat list of patches back into a full image or mask.
    Works for both grayscale masks (channels=None) and RGB images (channels=3).
    """
    H, W = patch_size, patch_size
    full_h, full_w = num_rows * H, num_cols * W
    shape  = (full_h, full_w) if channels is None else (full_h, full_w, channels)
    canvas = np.zeros(shape, dtype=np.float32)
    counts = np.zeros((full_h, full_w), dtype=np.float32)

    for idx, patch in enumerate(patches_list):
        i, j    = divmod(idx, num_cols)
        r0, c0  = i * H, j * W
        if channels is None:
            canvas[r0:r0+H, c0:c0+W] += patch
        else:
            canvas[r0:r0+H, c0:c0+W, :] += patch
        counts[r0:r0+H, c0:c0+W] += 1

    # Average overlapping regions (none with step=patch_size, but safe)
    if channels is None:
        canvas = np.divide(canvas, counts, where=counts > 0).astype(np.uint8)
        canvas = np.clip(canvas, 0, 1)
    else:
        canvas = np.divide(canvas, counts[..., None], where=counts[..., None] > 0).astype(np.uint8)
    return canvas

## 4. Load Model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model = torch.load(MODEL_PATH, weights_only=False)
model.eval()
model.to(device)
print('Model loaded.')

## 5. Prediction Pipeline

`predict_image` runs the full tile → infer → stitch pipeline for a single image path and saves both the predicted mask and the cropped RGB image.

The loop below runs it for every image matched by `IMAGE_GLOB`.

In [ ]:
os.makedirs(OUTPUT_MASK_DIR,  exist_ok=True)
os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)


def predict_image(image_path, model, device,
                  patch_size=256, batch_size=4):
    """Tile, predict, and stitch one image.

    Args:
        image_path : str  — path to the input GeoTIFF (or any PIL-readable image)
        model      : loaded DINOv2 segmentation model in eval mode
        device     : torch device
        patch_size : int  — tile size in pixels (must match training)
        batch_size : int  — tiles per forward pass

    Saves:
        <OUTPUT_MASK_DIR>/<stem>_mask.png   — binary predicted mask (0/255)
        <OUTPUT_IMAGE_DIR>/<stem>_image.png — cropped RGB image matching the mask
    """
    stem = os.path.splitext(os.path.basename(image_path))[0]
    print(f'\nProcessing: {stem}')

    # 1. Tile
    patches_rgb, num_rows, num_cols = tile_image(image_path, patch_size)
    n_patches = len(patches_rgb)
    print(f'  Grid: {num_rows} rows × {num_cols} cols = {n_patches} patches')

    # 2. Build dataloader over patches
    patch_ds     = PatchDataset(patches_rgb, patch_transform)
    patch_loader = DataLoader(patch_ds, batch_size=batch_size, shuffle=False)

    # 3. Run inference on every patch
    predicted_patches = []
    with torch.no_grad():
        for batch in tqdm(patch_loader, desc='  Predicting', leave=False):
            batch   = batch.to(device)
            outputs = model(batch)
            # Upsample logits to patch size, then argmax → binary
            upsampled = torch.nn.functional.interpolate(
                outputs.logits, size=(patch_size, patch_size),
                mode='bilinear', align_corners=False
            )
            preds = upsampled.argmax(dim=1).cpu().numpy()  # (B, H, W)
            predicted_patches.extend(preds)

    # 4. Stitch mask and RGB
    full_mask  = unpatchify(predicted_patches, patch_size, num_rows, num_cols, channels=None)
    full_image = unpatchify(list(patches_rgb),  patch_size, num_rows, num_cols, channels=3)

    # 5. Save outputs
    mask_path  = os.path.join(OUTPUT_MASK_DIR,  f'{stem}_mask.png')
    image_path_out = os.path.join(OUTPUT_IMAGE_DIR, f'{stem}_image.png')
    Image.fromarray((full_mask * 255).astype(np.uint8), mode='L').save(mask_path)
    Image.fromarray(full_image).save(image_path_out)
    print(f'  Saved mask  → {mask_path}')
    print(f'  Saved image → {image_path_out}')

In [ ]:
for image_path in image_paths:
    predict_image(image_path, model, device,
                  patch_size=PATCH_SIZE, batch_size=BATCH_SIZE)

print('\nAll images processed.')